In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()
using LorentzianSimplexSolver

  Activating project at `~/Documents/Work/effective-spinfoam/code/LorentzianSimplexSolver`
Precompiling project...
   5439.8 ms  ✓ LorentzianSimplexSolver
  1 dependency successfully precompiled in 7 seconds. 171 already precompiled.


In [2]:
# ------------------------------------------------------------
# 1. Precision choice (user-controlled)
# ------------------------------------------------------------
const ScalarT = Float64
#const ScalarT = BigFloat

if ScalarT === BigFloat
    LorentzianSimplexSolver.PrecisionUtils.set_big_precision!(256)
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(sqrt(eps(BigFloat)))
else
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(1e-8)
end

# ------------------------------------------------------------
# 2. Read simplices
# ------------------------------------------------------------
simplices = [[1, 2, 4, 8, 16], [1, 2, 4, 12, 16], [1, 2, 6, 8, 16], [1, 3, 4, 8, 16], [1, 2, 6, 14, 16], [1, 5, 6, 8, 16], [1, 3, 4, 12, 16], [1, 3, 7, 8, 16], [1, 3, 7, 15, 16], [1, 5, 7, 8, 16], [1, 5, 6, 14, 16], [1, 5, 7, 15, 16], [1, 2, 10, 12, 16], [1, 2, 10, 14, 16], [1, 9, 10, 12, 16], [1, 3, 11, 12, 16], [1, 3, 11, 15, 16], [1, 9, 11, 12, 16], [1, 9, 10, 14, 16], [1, 9, 11, 15, 16], [1, 5, 13, 14, 16], [1, 5, 13, 15, 16], [1, 9, 13, 14, 16], [1, 9, 13, 15, 16]]

ns = length(simplices)

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

Nverts = length(all_vertices)

# ------------------------------------------------------------
# 3. Read vertex coordinates
# ------------------------------------------------------------
vertex_coords = Dict{Int, Vector{ScalarT}}()    

coords_lines = [
    "0, 0, 0, 0",
    "0, 0, 0, 1",
    "0, 0, 1, 0",
    "0, 0, 1, 1",
    "0, 1, 0, 0",
    "0, 1, 0, 1",
    "0, 1, 1, 0",
    "0, 1, 1, 1",
    "1//2, 0, 0, 0",
    "1//2, 0, 0, 1",
    "1//2, 0, 1, 0",
    "1//2, 0, 1, 1",
    "1//2, 1, 0, 0",
    "1//2, 1, 0, 1",
    "1//2, 1, 1, 0",
    "1//2, 1, 1, 1",
]

for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [3]:
# ------------------------------------------------------------
# 4. Build geometry
# ------------------------------------------------------------
datasets = LorentzianSimplexSolver.GeometryTypes.GeometryDataset{ScalarT}[]

for (s, simplex) in enumerate(simplices)
    bdypoints = [vertex_coords[v] for v in simplex]
    # println("Processing simplex $s with vertices $(simplex)...")        
    ds = LorentzianSimplexSolver.GeometryPipeline.run_geometry_pipeline(bdypoints)
    push!(datasets, ds)
end

geom = LorentzianSimplexSolver.GeometryTypes.GeometryCollection(datasets);

In [4]:
# ------------------------------------------------------------
# 6. Connect simplices + face matching + gauge fixing
# ------------------------------------------------------------
if ns > 1
    LorentzianSimplexSolver.KappaOrientation.fix_kappa_signs!(simplices, geom)

    conn = LorentzianSimplexSolver.FourSimplexConnectivity.build_global_connectivity(simplices, geom)
    push!(geom.connectivity, conn)

    LorentzianSimplexSolver.FaceXiMatching.run_face_xi_matching(geom; sector=:ref)
    LorentzianSimplexSolver.GaugeFixingSU.run_su2_su11_gauge_fix(geom);
else
    sl2c = [geom.simplex[i].solgsl2c    for i in 1:ns]
    sgndet = [geom.simplex[i].sgndet    for i in 1:ns]
    geom.simplex[1].solgsl2c = LorentzianSimplexSolver.FaceXiMatching.update_sl2ctest(sl2c, sgndet)[1]
end;

In [5]:
LorentzianSimplexSolver.DefineSymbols.run_define_variables(geom);

In [6]:
sd, _ = LorentzianSimplexSolver.SolveVars.run_solver(geom);

In [7]:
S = LorentzianSimplexSolver.DefineAction.compute_action(geom);

In [8]:
using SymEngine
γ = LorentzianSimplexSolver.DefineAction.γsym()
vals = LorentzianSimplexSolver.ActionEvaluation.build_value_dict(sd, γ; γval=nothing);

In [9]:
S_val = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S, vals);
S_simpl = SymEngine.expand(S_val)

-1.1353381276742e-15 + 3.10597439719199e-15*im + (-1.84574577843932e-15 - 6.28318530717958*im)*gamma^(-1)

In [ ]:
dS = LorentzianSimplexSolver.EOMsHessian.compute_EOMs(S, sd)
dS = LorentzianSimplexSolver.EOMsHessian.check_EOMs(dS, sd; γ=1);

In [ ]:
g_vars = geom.varias[:g_var]
z_vars = geom.varias[:z_var]
j_vars = geom.varias[:j_var]
xi_vars = geom.varias[:xi_var]
vars = vcat(g_vars, z_vars, xi_vars, j_vars)
H_sym = LorentzianSimplexSolver.EOMsHessian.compute_Hessian_block_half(S, vars)
H_ref_eval = LorentzianSimplexSolver.EOMsHessian.evaluate_hessian_block(H_sym, sd; γ = 1);

In [ ]:
using LinearAlgebra
eigenvalues = eigvals(H_ref_eval);

In [ ]:
vals = sort(eigenvalues, by=abs, rev=true);

In [ ]:
Sregge_num, Sregge_symbol = LorentzianSimplexSolver.ReggeAction.run_Regge_action(geom, γ);

In [ ]:
Sregge_symbol

-0.5942407033369*j_113*gamma - 0.594240703336901*j_115*gamma - 4.44089209850063e-16*j_123*gamma + 5.55111512312578e-16*j_125*gamma + 4.44089209850063e-16*j_134*gamma - 6.66133814775094e-16*j_142*gamma - 2.22044604925031e-16*j_153*gamma + 0.361358596793957*j_212*gamma + 0.361358596793956*j_214*gamma - 1.77635683940025e-15*j_235*gamma + 4.71844785465692e-16*j_243*gamma - 4.9960036108132e-16*j_254*gamma + 0.361358596793959*j_313*gamma - 0.594240703336901*j_315*gamma + 8.88178419700125e-16*j_345*gamma - 0.594240703336902*j_412*gamma - 0.594240703336904*j_414*gamma + 0.361358596793959*j_513*gamma - 0.594240703336903*j_515*gamma